In [ ]:
import os
import sys, pathlib
import time
from itertools import chain, combinations, product

import numpy as np
import pandas as pd
from tqdm import tqdm
import joblib

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# =========================================================
# Paths
# =========================================================

PROJECT_ROOT = pathlib.Path().resolve()
while PROJECT_ROOT.name != "HAB_Phase1":
    PROJECT_ROOT = PROJECT_ROOT.parent
    if PROJECT_ROOT == PROJECT_ROOT.parent:
        raise RuntimeError("Could not find HAB_Phase1 project root")

sys.path.insert(0, str(PROJECT_ROOT))

from config import DATA_PATH, OUTPUT_DIRS

# DATA_PATH  = "Data/UCDavis_Data_for_Analysis2_220days_ToxinProdMicrocystis.csv"
# OUTPUT_DIR = "C:/Temp/FC_4Models_Results"

OUTPUT_DIR = str(OUTPUT_DIRS["01"])  # change for each notebook
DATA_PATH  = str(DATA_PATH)


def setup_output_dirs():
    subdirs = [
        "figures/model_comparison",
        "figures/confusion_matrix",
        "tables",
        "logs",
        "models/best_models",
        "models/configurations",
    ]
    for subdir in subdirs:
        os.makedirs(os.path.join(OUTPUT_DIR, subdir), exist_ok=True)


setup_output_dirs()

# Convenience path variables
TABLES_DIR      = os.path.join(OUTPUT_DIR, "tables")
BEST_MODELS_DIR = os.path.join(OUTPUT_DIR, "models", "best_models")
CONFIGS_DIR     = os.path.join(OUTPUT_DIR, "models", "configurations")

RESULT_FILE = os.path.join(TABLES_DIR, "results_with_hyperparams_combinations.csv")

# =========================================================
# Load Data
# =========================================================
df = pd.read_csv(DATA_PATH)

TARGET_COLUMN = "ToxinProdc_Microcystis_CC"
THRESHOLD     = 4000

# =========================================================
# Feature Columns  (17 features — single source of truth)
# =========================================================
feature_columns = [
    "Q", "Q_2w", "Q_1m",
    "SpecCond.", "Salinity", "NTU",
    "OrgC", "Total_OrgC",
    "NH4", "NO3", "OrgN",
    "PO4", "TP",
    "WaterTemp", "DO", "pH", "TDS",
]

# Verify every feature is present in the CSV
missing_cols = [f for f in feature_columns if f not in df.columns]
if missing_cols:
    raise ValueError(
        f"The following features are not found in the CSV:\n  {missing_cols}\n"
        f"Available columns: {df.columns.tolist()}"
    )

print(f"Feature columns ({len(feature_columns)}): {feature_columns}")

# =========================================================
# Build X and y
# =========================================================
X = df[feature_columns].copy()
y = np.where(df[TARGET_COLUMN] <= THRESHOLD, 0, 1)

print(f"\nDataset shape : X={X.shape}, y={y.shape}")
print(f"Class distribution: 0={np.sum(y == 0)}, 1={np.sum(y == 1)}")

# =========================================================
# 70 / 30 Stratified Train-Test Split
# =========================================================
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y,
)

print(f"\nTrain size : {X_train_raw.shape[0]} samples")
print(f"Test  size : {X_test_raw.shape[0]}  samples")
print(f"Train class dist: 0={np.sum(y_train == 0)}, 1={np.sum(y_train == 1)}")
print(f"Test  class dist: 0={np.sum(y_test  == 0)}, 1={np.sum(y_test  == 1)}")

# =========================================================
# Fit Fresh StandardScaler on Training Data Only
# =========================================================
global_scaler = StandardScaler()
global_scaler.fit(X_train_raw)          # fit on train only

# Full scaled arrays (all 17 features); columns sliced per combination later
X_train_full_scaled = global_scaler.transform(X_train_raw)
X_test_full_scaled  = global_scaler.transform(X_test_raw)

print(f"\nFresh StandardScaler fitted on {X_train_raw.shape[0]} training samples.")

# =========================================================
# FEATURE COMBINATION MODE
# =========================================================
COMBINATION_MODE        = "manual"  # "manual" or "auto"
MAX_COMBINATIONS_TO_RUN = None      # None = full run

mandatory_groups = [
    ["Q", "Q_2w", "Q_1m"],
    ["SpecCond.", "Salinity", "NTU"],
    ["OrgC", "Total_OrgC"],
    ["NH4", "NO3", "OrgN"],
    ["PO4", "TP"],
]

remaining_features = ["WaterTemp", "DO", "pH", "TDS"]

min_features = 9
max_features = 14

# =========================================================
# FEATURE COMBINATIONS
# =========================================================
if COMBINATION_MODE == "manual":

    feature_combinations = [
        ['Q', 'Q_1m', 'SpecCond.', 'OrgC', 'OrgN', 'PO4',
         'WaterTemp', 'DO', 'pH'],

        ['Q', 'SpecCond.', 'OrgC', 'Total_OrgC', 'NH4',
         'OrgN', 'PO4', 'WaterTemp', 'TDS'],

        ['Q_2w', 'Q_1m', 'OrgC', 'OrgN', 'PO4', 'TP',
         'WaterTemp', 'DO', 'pH', 'TDS'],

        ['Q', 'Q_2w', 'SpecCond.', 'Salinity', 'OrgC',
         'Total_OrgC', 'OrgN', 'PO4', 'WaterTemp',
         'DO', 'pH', 'TDS'],

        ['Q', 'Q_2w', 'Q_1m', 'SpecCond.', 'Salinity',
         'NTU', 'OrgC', 'Total_OrgC', 'NH4', 'NO3',
         'OrgN', 'PO4', 'TP', 'WaterTemp', 'DO',
         'pH', 'TDS']
    ]

else:

    feature_combinations = []

    for size in range(min_features, max_features + 1):

        mandatory_combinations = [
            list(chain.from_iterable(
                combinations(group, r)
                for r in range(1, min(3, len(group) + 1))
            ))
            for group in mandatory_groups
        ]

        for mandatory_selection in product(*mandatory_combinations):

            mandatory_flat  = list(chain.from_iterable(mandatory_selection))
            remaining_slots = size - len(mandatory_flat)

            if remaining_slots < 0:
                continue

            for remaining in combinations(remaining_features, remaining_slots):
                feature_combinations.append(mandatory_flat + list(remaining))

        if MAX_COMBINATIONS_TO_RUN is not None:
            feature_combinations = feature_combinations[:MAX_COMBINATIONS_TO_RUN]
            break

# =========================================================
# VALIDATE FEATURE COMBINATIONS AGAINST feature_columns
# =========================================================
print("\nValidating feature combinations against the 17-feature column list...")

for combo_idx, combo in enumerate(feature_combinations, start=1):
    missing = [f for f in combo if f not in feature_columns]
    if missing:
        raise ValueError(
            f"Feature combination {combo_idx} contains features not found "
            f"in the 17-feature list:\n"
            f"  Missing:     {missing}\n"
            f"  Combination: {combo}\n"
            f"  Available:   {feature_columns}"
        )

print(f"All {len(feature_combinations)} feature combinations validated successfully.")

# =========================================================
# TOTAL COMBINATIONS
# =========================================================
print("\n===================================================")
print(f"Total Feature Combinations : {len(feature_combinations)}")
print(f"Mode                       : {COMBINATION_MODE}")
print("===================================================")

# =========================================================
# MODELS & HYPERPARAMETER GRIDS
# =========================================================
models_config = {
    # ------------------------------------------------------------------
    # Logistic Regression
    #  n_iter = 500
    # ------------------------------------------------------------------
    "Logistic Regression": {
        "model": LogisticRegression(random_state=42, max_iter=1000),
        "params": [
            {
                "penalty":       ["l2"],
                "C":             [0.0001, 0.001, 0.01, 0.1, 1, 10, 100, 1000],
                "solver":        ["lbfgs"],
                "tol":           [0.0001, 0.001, 0.01, 0.1],
                "fit_intercept": [True, False],
                "class_weight":  [None, "balanced"],
            },
            {
                "penalty":       ["l1"],
                "C":             [0.0001, 0.001, 0.01, 0.1, 1, 10, 100, 1000],
                "solver":        ["liblinear"],
                "tol":           [0.0001, 0.001, 0.01, 0.1],
                "fit_intercept": [True, False],
                "class_weight":  [None, "balanced"],
            },
            {
                "penalty":       ["l1", "l2"],
                "C":             [0.0001, 0.001, 0.01, 0.1, 1, 10, 100, 1000],
                "solver":        ["saga"],
                "tol":           [0.0001, 0.001, 0.01, 0.1],
                "fit_intercept": [True, False],
                "class_weight":  [None, "balanced"],
            },
            {
                "penalty":       ["elasticnet"],
                "C":             [0.0001, 0.001, 0.01, 0.1, 1, 10, 100, 1000],
                "solver":        ["saga"],
                "l1_ratio":      [0.1, 0.25, 0.5, 0.75, 0.9],
                "tol":           [0.0001, 0.001, 0.01, 0.1],
                "fit_intercept": [True, False],
                "class_weight":  [None, "balanced"],
            },
        ],
        "n_iter": 500, #500
    },

    # ------------------------------------------------------------------
    # Decision Tree
    # n_iter = 2000
    # ------------------------------------------------------------------
    "Decision Tree": {
        "model": DecisionTreeClassifier(random_state=42),
        "params": {
            "criterion":         ["gini", "entropy"],
            "splitter":          ["best", "random"],
            "max_depth":         [None, 5, 10, 20, 30, 40],
            "min_samples_split": [2, 5, 10, 20],
            "min_samples_leaf":  [1, 2, 5, 10],
            "max_features":      [None, "sqrt", "log2", 0.5],
            "ccp_alpha":         [0.0, 0.001, 0.005, 0.01, 0.05],
            "max_leaf_nodes":    [None, 10, 20, 57, 100],
            "class_weight":      [None, "balanced"],
        },
        "n_iter": 2000, #2000
    },

    # ------------------------------------------------------------------
    # Random Forest
    # n_iter = 2000
    # ------------------------------------------------------------------
    "Random Forest": {
        "model": RandomForestClassifier(random_state=42),
        "params": {
            "n_estimators":              [50, 100, 200, 300],
            "criterion":                 ["gini", "entropy", "log_loss"],
            "max_depth":                 [None, 4, 5, 6, 8, 10, 12, 14, 16, 20],
            "min_samples_split":         [2, 5, 10],
            "min_samples_leaf":          [1, 2, 5, 8],
            "max_features":              ["sqrt", "log2", None, 0.3],
            "class_weight":              [None, "balanced", "balanced_subsample"],
            "ccp_alpha":                 [0.0, 0.0001, 0.001, 0.01],
            "min_impurity_decrease":     [0.0, 0.005, 0.01, 0.02],
            "min_weight_fraction_leaf":  [0.0, 0.01, 0.02],
            "max_leaf_nodes":            [None, 20, 57, 100],
            "bootstrap":                 [True, False],
        },
        "n_iter": 2000, #2000
    },

    # ------------------------------------------------------------------
    # XGBoost
    # n_iter = 2000
    # ------------------------------------------------------------------
    "XGBoost": {
        "model": XGBClassifier(random_state=42, eval_metric="logloss",
                               verbosity=0, n_jobs=1),
        "params": {
            "n_estimators":       [50, 100, 200, 300],
            "max_depth":          [2, 4, 6, 8, 10, 12, 14, 16],
            "learning_rate":      [0.001, 0.01, 0.05, 0.1],
            "subsample":          [0.6, 0.8, 1.0],
            "colsample_bytree":   [0.5, 0.6, 0.8, 1.0],
            "colsample_bylevel":  [0.6, 0.8, 1.0],
            "colsample_bynode":   [0.6, 0.75, 1.0],
            "gamma":              [0, 0.1, 0.2, 0.3],
            "min_child_weight":   [1, 3, 5],
            "reg_alpha":          [0, 0.01, 0.1],
            "reg_lambda":         [1, 1.5, 2],
            "scale_pos_weight":   [1, 2, 5],
            "max_delta_step":     [0, 1, 5],
        },
        "n_iter": 2000, #2000
    },
}

# =========================================================
# RESULTS FILE INIT
# =========================================================
if not os.path.exists(RESULT_FILE):
    pd.DataFrame(columns=[
        "Feature Combination Index",
        "Model",
        "Features",
        "Best Parameters",
        "Train Accuracy",
        "Test Accuracy",
        "Train Precision",
        "Test Precision",
        "Train Recall",
        "Test Recall",
        "Train F1",
        "Test F1",
    ]).to_csv(RESULT_FILE, index=False)

# =========================================================
# TRAINING LOOP
# =========================================================
for idx, feature_combination in enumerate(
    tqdm(feature_combinations, desc="Feature Combinations"),
    start=1
):

    start_time = time.time()

    print(f"\n=== Combination {idx}/{len(feature_combinations)} ===")
    print(feature_combination)

    # =====================================================
    # SLICE SCALED ARRAYS to this combination's columns
    # =====================================================
    feature_indices = [feature_columns.index(f) for f in feature_combination]

    X_train_scaled = X_train_full_scaled[:, feature_indices]
    X_test_scaled  = X_test_full_scaled[:,  feature_indices]

    best_models = {}
    best_params = {}

    # =====================================================
    # MODEL TRAINING
    # =====================================================
    for model_name, config in models_config.items():

        search = RandomizedSearchCV(
            estimator=config["model"],
            param_distributions=config["params"],
            n_iter=config["n_iter"],
            cv=5,
            scoring="accuracy",
            n_jobs=-1,
            random_state=42,
        )

        search.fit(X_train_scaled, y_train)

        best_models[model_name] = search.best_estimator_
        best_params[model_name] = search.best_params_

    # =====================================================
    # EVALUATION + TRACK BEST MODEL
    # =====================================================
    best_test_accuracy = -1
    best_model_name    = None
    best_model_obj     = None

    for model_name, model in best_models.items():

        y_train_pred = model.predict(X_train_scaled)
        y_test_pred  = model.predict(X_test_scaled)

        result = {
            "Feature Combination Index": idx,
            "Model":           model_name,
            "Features":        feature_combination,
            "Best Parameters": best_params[model_name],
            "Train Accuracy":  accuracy_score(y_train, y_train_pred),
            "Test Accuracy":   accuracy_score(y_test,  y_test_pred),
            "Train Precision": precision_score(y_train, y_train_pred, zero_division=0),
            "Test Precision":  precision_score(y_test,  y_test_pred,  zero_division=0),
            "Train Recall":    recall_score(y_train, y_train_pred, zero_division=0),
            "Test Recall":     recall_score(y_test,  y_test_pred,  zero_division=0),
            "Train F1":        f1_score(y_train, y_train_pred, zero_division=0),
            "Test F1":         f1_score(y_test,  y_test_pred,  zero_division=0),
        }

        pd.DataFrame([result]).to_csv(
            RESULT_FILE,
            mode="a",
            header=False,
            index=False,
        )

        if result["Test Accuracy"] > best_test_accuracy:
            best_test_accuracy = result["Test Accuracy"]
            best_model_name    = model_name
            best_model_obj     = model

    # =====================================================
    # SAVE BEST MODEL & CONFIG FOR THIS COMBINATION
    # =====================================================
    joblib.dump(
        best_model_obj,
        os.path.join(BEST_MODELS_DIR, f"best_model_combination_{idx}.pkl"),
    )

    joblib.dump(
        {
            "combination_index":  idx,
            "model_name":         best_model_name,
            "features":           feature_combination,
            "best_params":        best_params[best_model_name],
            "feature_indices":    feature_indices,
            "best_test_accuracy": best_test_accuracy,
            # Scaler saved alongside so downstream scripts can reproduce
            # transforms without re-fitting
            "scaler":             global_scaler,
            "scaler_features":    feature_columns,
        },
        os.path.join(CONFIGS_DIR, f"best_model_config_combination_{idx}.pkl"),
    )

    print(f"  Best model : {best_model_name} | Test Accuracy: {best_test_accuracy:.4f}")
    print(f"  Completed  in {time.time() - start_time:.2f} sec")

print("\nAll processing completed.")